# Proof of Concept  Notebook B (Team Blue)

Runs in its own kernel, separate from Notebook A. Does NOT start the
competition — Notebook A does. Joins as `team_blue` and plays.

**Critical:** `SERVER_IP` / `SERVER_PORT` MUST match Notebook A exactly.

In [1]:
from pynqsim import SimulationClient

SERVER_IP = "172.20.166.171"   # <-- same server IP as Notebook A
SERVER_PORT = 9003             # <-- same server port as Notebook A

blue = SimulationClient(SERVER_IP, SERVER_PORT)
print("Team Blue connected to server!")

Team Blue connected to server!


In [2]:
# Team Blue joins with the join password the instructor gave you.
blue.join_competition(team_id="team_blue", password="blue123")
print("Joined as team_blue!")

Joined as team_blue!


In [3]:
state = blue.get_competition_state()
print(f"Current turn: {state.get('current_turn')}")
print(f"Scores: {state.get('scores', {})}")

Current turn: team_red
Scores: {'team_red': 0, 'team_blue': 0}


## Color names (6x5 grid → 15 colors)
MUST match the order the scene assigns `color_idx` (indices 0..14).

In [4]:
# 15 colors, in the exact order the scene assigns color_idx (0..14).
COLOR_NAMES = [
    "RED", "GRN", "BLU", "YEL", "ORG",
    "PUR", "BRN", "PNK", "CYN", "MAG",
    "LIM", "TEA", "GRY", "MAR", "NVY",
]

def color_name(ci):
    return COLOR_NAMES[ci] if (ci is not None and ci < len(COLOR_NAMES)) else str(ci)

## Normal play (`flip`)
Drives the BLUE arm. Wait until it's Blue's turn or the server raises "Not your
turn". Match = go again; mismatch = re-cover + turn passes.

In [ ]:
def flip(client, row, col):
    """Flip a card WITH memory-game scoring/turn messaging."""
    result = client.flip_card(row, col)
    if result.get("already_flipped"):
        print(f"Card ({row},{col}) already flipped")
        return result
    print(f">>> ({row},{col}) revealed: {color_name(result['color_idx'])}")
    mr = result.get("match_result")
    if mr:
        if mr.get("matched"):
            print("*** MATCH! +1 — same team goes again! ***")
        elif mr.get("turn_ended"):
            print("No match. Cards re-covered. Turn passes to the other team.")
    return result

flip(blue, 3, 3)   # first card
flip(blue, 5, 1) # second card

## `flip_raw()` — grab a cover with NO scoring / turn / re-cover
Calls the **`flip_raw` server action** (add the server snippet first). Skips
`record_flip`, so no scorekeeping, no turn check, and mismatched covers are NOT
teleported back — the cover stays in the discard pile. `robot_id` picks the arm:
**0=red (left cols), 1=blue (right cols)**, since one arm can't reach all 30 cards.

In [6]:
def flip_raw(client, row, col, robot_id=None):
    """Grab one cover: no scoring, no turn logic, no re-cover."""
    params = {"row": row, "col": col}
    if robot_id is not None:
        params["robot_id"] = robot_id
    result = client._request("flip_raw", params)
    if result.get("already_flipped"):
        print(f"({row},{col}) already flipped")
    else:
        print(f"grabbed ({row},{col}) -> {color_name(result.get('color_idx'))}")
    return result

flip_raw(blue,4,0)
flip_raw(blue,4,1)
flip_raw(blue,4,2)
flip_raw(blue,4,3)
flip_raw(blue,4,4)

grabbed (4,0) -> PNK
grabbed (4,1) -> NVY
grabbed (4,2) -> NVY
grabbed (4,3) -> CYN
grabbed (4,4) -> GRY


{'color_idx': 12, 'already_flipped': False, 'matched': False, 'status': 'ok'}

## Reset the board (instructor action)
This is an **admin** action, normally run from the Red (instructor) notebook.
Included here only if Blue's operator has the admin password.

In [7]:
# Admin action -- needs the admin password (same one the instructor set).
ADMIN_PASSWORD = "admin123"   # <-- must match the server's admin password

blue.admin_reset_board(password=ADMIN_PASSWORD)
state = blue.get_competition_state()
print("Board reset.")
print(f"Scores: {state.get('scores', {})}")
print(f"Current turn: {state.get('current_turn')}")

Board reset.
Scores: {'team_red': 0, 'team_blue': 0}
Current turn: team_red


## Cleanup

In [ ]:
blue.leave_competition()
print("Team Blue left.")